# Lab 3: Build a Search Agent

In this lab, we'll use the Azure AI Agent Service to create an agent that is able to retrieve information from documents stored in Azure AI Search, a vector database. This pattern is known as retrieval augmented generation or RAG. The documents that we'll be searching are health insurance policies.

#### Step 1: Load packages

In [ ]:
import os
import json
import urllib.request
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from azure.ai.agents.models import FunctionTool, ToolSet, AgentThreadCreationOptions

load_dotenv()


#### Step 2: Connect to your Microsoft Foundry project

In [ ]:
# Use AzureCliCredential (requires az login) for the agents API which needs token-based auth
credential = AzureCliCredential()

# Connecting to our Microsoft Foundry project
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=credential
)


#### Step 3: Connect to your Azure AI Search index

In [ ]:
# First enter the name of your search index

index_name="health-plan"
print(index_name)

This code retrieves the connection ID for your Azure AI Search resource, then defines and configures the search tool and its resources. It ensures your agent is set up to access the correct search index, enabling it to perform document retrieval using the Azure AI Agent Service SDK.

In [ ]:
# Get the AI Search connection details (including API key) from the project
conn_id = None
search_endpoint = None
search_api_key = None

for conn in project.connections.list():
    conn_type = str(getattr(conn, "type", ""))
    if "SEARCH" in conn_type.upper() or "CognitiveSearch" in conn_type:
        conn_full = project.connections.get(name=conn.name, include_credentials=True)
        search_endpoint = conn_full.as_dict().get("target", "").rstrip("/")
        search_api_key = conn_full.as_dict().get("credentials", {}).get("key")
        conn_id = conn.id
        print(f"Found AI Search connection: {conn.name}")
        break

if not search_endpoint or not search_api_key:
    raise ValueError("Could not retrieve AI Search connection details.")

# Define a Python function that performs the keyword search directly
def search_health_plan_index(query: str) -> str:
    """
    Searches the health-plan Azure AI Search index for documents relevant to the given query.
    Returns the top matching document chunks as a JSON string.

    :param query: The search query string.
    :return: JSON string with the top search results (title and chunk).
    """
    url = f"{search_endpoint}/indexes/{index_name}/docs/search?api-version=2023-11-01"
    body = json.dumps({
        "search": query,
        "queryType": "simple",
        "top": 5,
        "select": "title,chunk"
    }).encode()
    req = urllib.request.Request(
        url, data=body,
        headers={"api-key": search_api_key, "Content-Type": "application/json"},
        method="POST"
    )
    resp = urllib.request.urlopen(req)
    data = json.loads(resp.read())
    results = [{"title": d.get("title", ""), "chunk": d.get("chunk", "")} for d in data.get("value", [])]
    return json.dumps(results, ensure_ascii=False)

# Wrap the function in a FunctionTool so the agent can call it
search_functions = FunctionTool(functions={search_health_plan_index})
print("Search tool ready.")


#### Step 4: Define the search agent

In this step, you will define and create the search agent using the Azure AI Agent Service SDK. The agent is configured with the GPT-4.1 model, a descriptive name, instructions for its behavior, and the search tool and resources you set up previously. This setup enables the agent to process user queries and retrieve relevant information from your Azure AI Search index, making it capable of intelligent, document-grounded search.

In [ ]:
toolset = ToolSet()
toolset.add(search_functions)

search_agent = project.agents.create_agent(
    model=os.getenv("CHAT_MODEL"),
    name="search-agent",
    instructions="You are a helpful agent that is an expert at searching health plan documents. Use the search_health_plan_index tool to retrieve relevant information, then summarize what you find.",
    toolset=toolset
)
print(f"Created search agent, ID: {search_agent.id}")


#### Step 5: Chat with the search agent

In this step, you'll interact with your search agent by sending it a user query and processing its response. The code demonstrates how to:
- Create a conversation thread with an initial user message
- Run the agent to process the query using the Azure AI Agent Service SDK
- Retrieve and display the agent's response, which is grounded in the indexed health plan documents
- Clean up by deleting the agent after use

This hands-on interaction shows how retrieval-augmented generation (RAG) enables your agent to provide accurate, document-based answers to natural language questions.

In [ ]:
# The name of the health plan we want to search for
plan_name = 'Northwind Standard'

# Create a thread and add the user message
thread = project.agents.threads.create()
project.agents.messages.create(
    thread_id=thread.id,
    role="user",
    content=f"Tell me about the {plan_name} plan."
)

# Manually poll and dispatch FunctionTool calls
run = project.agents.runs.create(thread_id=thread.id, agent_id=search_agent.id)

import time
while run.status in ("queued", "in_progress", "requires_action"):
    time.sleep(1)
    run = project.agents.runs.get(thread_id=thread.id, run_id=run.id)

    if run.status == "requires_action":
        tool_outputs = []
        for tool_call in run.required_action.submit_tool_outputs.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"Agent calling: {fn_name}({list(fn_args.keys())})")
            if fn_name == "search_health_plan_index":
                result = search_health_plan_index(**fn_args)
            else:
                result = json.dumps({"error": f"Unknown function: {fn_name}"})
            tool_outputs.append({"tool_call_id": tool_call.id, "output": result})
        project.agents.runs.submit_tool_outputs(
            thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs
        )

print(f"Run finished with status: {run.status}")
if run.status == "failed":
    print(f"Error: {run.last_error}")

# Delete the agent when done
project.agents.delete_agent(search_agent.id)
print("Deleted agent")

# Print the agent's response
messages = list(project.agents.messages.list(thread_id=thread.id))
for msg in messages:
    role = str(getattr(msg, "role", ""))
    if "agent" in role.lower() or "assistant" in role.lower():
        for part in (msg.content or []):
            # Handle both MessageTextContent objects and plain dicts
            text_val = None
            if hasattr(part, "text"):
                text_val = getattr(part.text, "value", None) or (part.text.get("value") if isinstance(part.text, dict) else None)
            elif isinstance(part, dict) and part.get("type") == "text":
                text_val = part.get("text", {}).get("value")
            if text_val:
                print(f"\nAgent: {text_val}")
                break
        break
